# Notebook 5 – Data Validation

## 1. What is data validation?
Validation checks whether data meets expected types, formats, ranges, categories, null rules, dates, references, and business constraints. Validation flags problems; it does not always fix them automatically.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [25, -3, 42, 150],
    "Salary": [30000, -500, 45000, 60000],
    "Percentage": [10, 105, 0, 25],
    "Gender": ["Female", "Unknown", "Male", "Female"],
    "DOB": ["2000-01-01", "1990-05-01", "2020-01-01", "1980-02-01"],
    "JoiningDate": ["2022-01-01", "2020-01-01", "2019-01-01", "2021-01-01"]
})
display(df)

,Age,Salary,Percentage,Gender,DOB,JoiningDate
0,25,30000,10,Female,2000-01-01,2022-01-01
1,-3,-500,105,Unknown,1990-05-01,2020-01-01
2,42,45000,0,Male,2020-01-01,2019-01-01
3,150,60000,25,Female,1980-02-01,2021-01-01


## 2. Range, type, and format validation
- Range: values must be within allowed limits.
- Type: values must have the expected data type.
- Format: values must follow an expected representation such as a date or identifier pattern.

**Example:** Age and salary should not be negative under the illustrative rules; percentage should be between 0 and 100.

In [2]:
df["DOB"] = pd.to_datetime(df["DOB"], errors="coerce")
df["JoiningDate"] = pd.to_datetime(df["JoiningDate"], errors="coerce")

checks = pd.DataFrame({
    "age_valid": df["Age"].between(0, 120),
    "salary_nonnegative": df["Salary"] >= 0,
    "percentage_valid": df["Percentage"].between(0, 100),
    "dob_is_date": df["DOB"].notna(),
    "joining_date_is_date": df["JoiningDate"].notna()
})
display(checks)
print("Rows failing at least one rule:", (~checks.all(axis=1)).sum())

,age_valid,salary_nonnegative,percentage_valid,dob_is_date,joining_date_is_date
0,True,True,True,True,True
1,False,False,False,True,True
2,True,True,True,True,True
3,False,True,True,True,True


Rows failing at least one rule: 2


**Output explanation:** `True` means the rule passes. Age limit 120 is illustrative; production thresholds should be agreed with domain owners.

## 3. Category and null validation
Categories should belong to an accepted set. Required fields should not be null. Unknown categories may need review rather than silent replacement.

In [3]:
allowed_gender = {"Female", "Male", "Other"}
category_valid = df["Gender"].isin(allowed_gender)
required_fields_present = df[["Age", "DOB", "JoiningDate"]].notna().all(axis=1)
print("Invalid category rows:", df.index[~category_valid].tolist())
print("Rows missing required fields:", df.index[~required_fields_present].tolist())

Invalid category rows: [1]
Rows missing required fields: []


## 4. Business rule, date, referential, and constraint validation
Examples: joining date must not precede date of birth; foreign keys must exist in the parent table; a value must be unique if the business defines it as unique.

In [4]:
date_order_valid = df["JoiningDate"] >= df["DOB"]
print("Date-order violations:", df.index[~date_order_valid].tolist())

employees = pd.DataFrame({"employeeID": ["E1", "E2"]})
payroll = pd.DataFrame({"employeeID": ["E1", "E9"], "amount": [1000, 1200]})
referential_valid = payroll["employeeID"].isin(employees["employeeID"])
print("Payroll rows with missing employee reference:", payroll.index[~referential_valid].tolist())

# Example uniqueness constraint
ids = pd.Series(["E1", "E2", "E2"])
print("Duplicate ID positions:", ids.index[ids.duplicated(keep=False)].tolist())

Date-order violations: [2]
Payroll rows with missing employee reference: [1]
Duplicate ID positions: [1, 2]


**Output explanation:** Each check returns the records that violate a defined rule. Validation rules must reflect the real domain; for example, some businesses may allow negative adjustments or historical exceptions.